In [1]:
import requests
from scrapy import Selector
import pandas as pd

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd

In [3]:
# Set up Chrome options for Selenium
#Dynamic JavaScript-based websites cannot be scraped using 
# basic HTTP requests alone; browser automation tools like Selenium are required.
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--headless")  # Headless mode is used to run the browser without a graphical interface to improve performance.
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--no-sandbox")

#### The ChromeDriverManager is used to automatically download and manage the compatible ChromeDriver version, and the Service object initializes the driver for Selenium.

In [4]:
service = Service(ChromeDriverManager().install())

In [5]:
driver = webdriver.Chrome(service=service, options=chrome_options)

#### The driver.get() method is used to navigate the automated browser to the specified URL.

In [6]:
url = 'https://www.shopsy.in/smart-watches-online'
driver.get(url)

#### driver.page_source retrieves the complete HTML content of the dynamically rendered webpage.

In [7]:
page_source = driver.page_source

In [8]:
sel = Selector(text=page_source)

#### This list contains pagination URLs to scrape product data from multiple pages instead of only the first page, ensuring comprehensive data collection.

In [9]:
# List of URLs
links =['https://www.shopsy.in/smart-watches-online',
        'https://www.shopsy.in/smart-watches-online/pr?page=2',
        'https://www.shopsy.in/smart-watches-online/pr?page=3',
        'https://www.shopsy.in/smart-watches-online/pr?page=4',
        'https://www.shopsy.in/smart-watches-online/pr?page=5',
        'https://www.shopsy.in/smart-watches-online/pr?page=6',
        'https://www.shopsy.in/smart-watches-online/pr?page=7',
        'https://www.shopsy.in/smart-watches-online/pr?page=8',
        'https://www.shopsy.in/smart-watches-online/pr?page=9',
        'https://www.shopsy.in/smart-watches-online/pr?page=10',
        'https://www.shopsy.in/smart-watches-online/pr?page=11',
        'https://www.shopsy.in/smart-watches-online/pr?page=12',
        'https://www.shopsy.in/smart-watches-online/pr?page=13',
        'https://www.shopsy.in/smart-watches-online/pr?page=14',
        'https://www.shopsy.in/smart-watches-online/pr?page=15',
        'https://www.shopsy.in/smart-watches-online/pr?page=16',
          ]

# OR

#### Since the website uses pagination, I generated URLs for multiple pages to scrape a larger dataset and avoid missing product records.

#### To handle pagination efficiently, I generated the URLs dynamically using a loop. This approach improves scalability and avoids hardcoding multiple links.

In [10]:
links = []
for i in range(1, 17):
    if i == 1:
        links.append("https://www.shopsy.in/smart-watches-online")
    else:
        links.append(f"https://www.shopsy.in/smart-watches-online/pr?page={i}")

In [11]:
# List to store scraped data
data = []


#### In this scraping process, I handled pagination dynamically, sent HTTP requests to each page, parsed the HTML using Scrapy Selector, extracted structured product details using CSS selectors, stored them in dictionary format, and implemented exception handling to ensure robustness.

In [12]:
for link in links:
    try:
        # Make a request to each URL
        response = requests.get(link, headers={"User-Agent": "Mozilla/5.0"})
        response.raise_for_status()  # Check if the request was successful
        
        # Parse the HTML response
        sel = Selector(text=response.text)
        
        # Extract data
        product_names = sel.css("div.css-175oi2r.r-1habvwh.r-eqz5dr.r-1h0z5md > div > div.css-175oi2r.r-1habvwh.r-eqz5dr.r-1h0z5md.r-cnw61z.r-1ss6j8a.r-1hvjb8t > div.css-146c3p1.r-dnmrzs.r-1udh08x.r-1udbk01.r-3s2u2q.r-1iln25a > span::text").getall()
        offer = sel.css("div > div > div.css-175oi2r.r-1habvwh.r-eqz5dr.r-1h0z5md > div > div.css-175oi2r.r-1habvwh.r-eqz5dr.r-1h0z5md.r-cnw61z.r-1ss6j8a.r-1hvjb8t > div:nth-child(2) > div:nth-child(1) > div:nth-child(1) > div::text").getall()
        product_price = sel.css("div.css-175oi2r.r-1rngwi6 > div.css-146c3p1.r-1h7g6bg.r-1vgyyaa.r-1rsjblm.r-142tt33.r-11wrixw::text").getall()
        discount_price = sel.css("div > div > div > div.css-175oi2r.r-1habvwh.r-eqz5dr.r-1h0z5md.r-cnw61z.r-1ss6j8a.r-1hvjb8t > div:nth-child(2) > div:nth-child(1) > div.css-175oi2r.r-1rngwi6 > div.css-146c3p1.r-cqee49.r-1vgyyaa.r-1rsjblm.r-13hce6t::text").getall()
        reviwe = sel.css("div.css-175oi2r.r-1awozwy.r-13awgt0.r-18u37iz.r-1h0z5md.r-f1odvy > div > div:nth-child(2) > div > div::text").getall()
        retings = sel.css("div.css-175oi2r.r-1awozwy.r-13awgt0.r-18u37iz.r-1h0z5md.r-f1odvy > div > div.css-175oi2r.r-1awozwy.r-cdmcib.r-jwli3a.r-6koalj.r-18u37iz.r-1enofrn.r-1ss6j8a > div::text").getall()
        deliverry = sel.css("div.css-175oi2r.r-1habvwh.r-eqz5dr.r-1h0z5md.r-cnw61z.r-1ss6j8a.r-1hvjb8t > div.css-175oi2r.r-18u37iz.r-1d09ksm.r-1w6e6rj.r-kc8jnq > div::text").getall()
        #ratings = sel.css("._5OesEi::text").getall()
        #warranty = sel.css("ul > li:nth-child(6)::text").getall()

        # Store data in list
        for i in range(len(product_names)):
            data.append({
                "Product Name": product_names[i] if i < len(product_names) else None,
                "Offer": offer[i] if i < len(offer) else None,
                "Product_price": product_price[i] if i < len(product_price) else None,
                "Discount_price": discount_price[i] if i < len(discount_price) else None,
                "Reviwe": reviwe [i] if i < len(reviwe) else None,
                "Reting": retings[i] if i < len(retings) else None,
                "Delivery":deliverry[i] if i < len(deliverry) else None,
            })
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {link}: {e}")


In [13]:
# Convert data to DataFrame and save to CSV
df = pd.DataFrame(data)
df.to_csv("scraped_data.csv", index=False)
    

df.head()

,Product Name,Offer,Product_price,Discount_price,Reviwe,Reting,Delivery
0,Akash Traders Bluetooth Calling Smart watch T5...,81% off,"1,999",₹367,(696),3.5,Bank Offer
1,SHIVANA NEW UPDATED T500 CALLING Z-13 Smartwatch,88% off,"2,999",₹357,(5),4.2,Bank Offer
2,"Gamesir I8 Pro pink Smartwatch with Bluetooth,...",70% off,"1,999",₹587,(23),3.9,Hot Deal
3,Dfuli i20 Ultra Android Smart Watch- 7in1 Comb...,71% off,"2,199",₹628,(18),4,Only few left
4,"Rls T500Ultra Max""HD Display Bluetooth Calling...",74% off,"1,499",₹382,(3),4.7,Only few left


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 640 entries, 0 to 639
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Product Name    640 non-null    object
 1   Offer           640 non-null    object
 2   Product_price   640 non-null    object
 3   Discount_price  640 non-null    object
 4   Reviwe          509 non-null    object
 5   Reting          509 non-null    object
 6   Delivery        640 non-null    object
dtypes: object(7)
memory usage: 35.1+ KB


In [15]:
null_counts = df[['Product Name','Offer','Product_price','Discount_price','Reviwe', 'Reting', 'Delivery']].isnull().sum()

print("Number of null values in each variable:")
print(null_counts)

Number of null values in each variable:
Product Name        0
Offer               0
Product_price       0
Discount_price      0
Reviwe            131
Reting            131
Delivery            0
dtype: int64
